# Ансамбль F: два ResNet50 подряд

Две модели конфигурации F, seed 7 затем seed 42. Общий кэш, без EMA и SWA. На инференсе TTA8 и среднее softmax. Test не участвует в обучении.

In [ ]:
import time
PIPELINE_START = time.perf_counter()

import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, confusion_matrix
from torch.utils.data import TensorDataset

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src import train_f as tf

assert torch.cuda.is_available(), "CUDA is not available"
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = "cuda"
EPOCHS = 15
BATCH = 64
SEEDS = (7, 42)
LOG_BEST = {7: 13, 42: 12}
N_TEST = 8299

## Данные и кэш

Train и val читаются один раз. Искажённый val берётся из `outputs/val_corrupted.pt`. Test в этом блоке не загружается.

In [ ]:
classes = pd.read_csv(tf.DATA_DIR / "classes.csv").sort_values("label")
class_names = classes.class_name.tolist()
assert class_names and len(class_names) == 20
train_table = pd.read_csv(tf.DATA_DIR / "train.csv", dtype={"id": str})
val_table = pd.read_csv(tf.DATA_DIR / "val.csv", dtype={"id": str})
assert set(train_table.label.unique()) == set(range(20))
assert set(val_table.label.unique()) == set(range(20))
train_u8 = tf.load_uint8(train_table, "train")
val_u8 = tf.load_uint8(val_table, "val")
train_y = tf._labels_of(train_table)
val_y = tf._labels_of(val_table)
blob = torch.load(tf.CORRUPT_PATH, map_location="cpu", weights_only=False)
assert blob["seed"] == tf.CORRUPT_SEED
assert torch.equal(blob["labels"], val_y)
corrupt_u8 = blob["images"]
assert tuple(train_u8.shape[1:]) == (3, 256, 256)
assert tuple(val_u8.shape) == (len(val_table), 3, 256, 256)
print(f"cache train/val/corrupt {len(train_table)}/{len(val_table)}/{len(corrupt_u8)} | test not loaded", flush=True)

## Аугментации

Конфигурация F, вход 256. На обучении: flips, поворот на 90°, ColorJitter, blur p=0.2, downscale-upscale p=0.2 с antialias, JPEG или шум p=0.1, invert p=0.03. На val только приведение к 256 и нормализация ImageNet. Функции те же, что воспроизводят траекторию F.

In [ ]:
tf._RN_MEAN = torch.tensor(tf.RN_MEAN, device=DEVICE).view(1, 3, 1, 1)
tf._RN_STD = torch.tensor(tf.RN_STD, device=DEVICE).view(1, 3, 1, 1)
print("aug F | img 256 | ImageNet norm", flush=True)

## Модель

ResNet50 с весами IMAGENET1K_V1. Разморожены layer3, layer4 и голова. Learning rate: layer3 5e-5, layer4 1e-4, голова 1e-3. AdamW, cosine на 15 эпох, label smoothing 0.1, batch 64, AMP. EMA и SWA нет.

In [ ]:
def make_model():
    model = tf.build_backbone("IMAGENET1K_V1")
    model.train()
    tf.freeze_running_bn(model)
    return model


def make_optimizer(model):
    return tf.optimizer_of(model, 5e-5, 1e-4, 1e-3)

## Обучение

Перед каждым seed заново ставятся seed Python, NumPy и PyTorch, генератор DataLoader и новая модель. Лучшая эпоха — максимум (чистый + искажённый) / 2. При равенстве остаётся более ранняя. По логу раунда 3 это эпоха 13 для seed 7 и эпоха 12 для seed 42. Файл `resnet50_F_seed42.pt` сохранён с эпохи 11: тот выбор шёл по чистому val, не по среднему.

In [ ]:
def train_epoch(model, loader, optimizer, scaler, criterion, batch_limit=None):
    model.train()
    tf.freeze_running_bn(model)
    optimizer.zero_grad(set_to_none=True)
    seen = 0
    for images, labels in loader:
        if batch_limit is not None and seen >= batch_limit:
            break
        images = tf.gpu_augment(images.to(DEVICE, non_blocking=True), cfg="F", img_size=256)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        seen += 1
    return seen


def fit_f(seed, train_u8, train_y, val_u8, val_y, corrupt_u8):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    model = make_model()
    optimizer = make_optimizer(model)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    train_loader = tf.make_loader(TensorDataset(train_u8, train_y), True, seed, BATCH)
    val_loader = tf.make_loader(TensorDataset(val_u8, val_y), False, seed, BATCH)
    corrupt_loader = tf.make_loader(TensorDataset(corrupt_u8, val_y), False, seed, BATCH)
    best = None
    best_state = None
    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        train_epoch(model, train_loader, optimizer, scaler, criterion)
        clean, sk_clean = tf.accuracy_of(model, val_loader)
        corrupt, sk_corrupt = tf.accuracy_of(model, corrupt_loader)
        assert abs(clean - sk_clean) < 1e-12 and abs(corrupt - sk_corrupt) < 1e-12
        scheduler.step()
        torch.cuda.synchronize()
        score = (clean + corrupt) / 2.0
        if best is None or score > best["score"]:
            best = {"epoch": epoch, "clean": clean, "corrupt": corrupt, "score": score}
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(
            f"seed {seed} | epoch {epoch:2d} | {clean:.4f}/{corrupt:.4f} | {time.perf_counter() - t0:.1f}s",
            flush=True,
        )
    print(
        f"seed {seed} | best epoch {best['epoch']} | {best['clean']:.4f}/{best['corrupt']:.4f}",
        flush=True,
    )
    if best["epoch"] != LOG_BEST[seed]:
        raise RuntimeError(
            f"seed {seed}: rule picked epoch {best['epoch']}, round-3 log picks {LOG_BEST[seed]}"
        )
    del model, optimizer, scaler, scheduler, criterion
    torch.cuda.empty_cache()
    return best, best_state

## Seed 7

Первая модель. После цикла на CPU остаётся только клон лучших весов.

In [ ]:
best7, state7 = fit_f(7, train_u8, train_y, val_u8, val_y, corrupt_u8)
torch.cuda.empty_cache()

## Seed 42

Вторая модель строится заново, со своим seed. Веса seed 7 уже лежат на CPU и в этом цикле не используются.

In [ ]:
best42, state42 = fit_f(42, train_u8, train_y, val_u8, val_y, corrupt_u8)
torch.cuda.empty_cache()

## Оценка

TTA8 — четыре поворота и отражение, среднее softmax. Сначала каждая модель на чистом и искажённом val, затем ансамбль. Ниже матрица ошибок ансамбля на чистом val, топ-5 пар и шесть кадров.

In [ ]:
@torch.no_grad()
def tta_probs(model, images):
    model.eval()
    chunks = []
    for start in range(0, images.shape[0], BATCH):
        base = tf.gpu_eval(images[start:start + BATCH].to(DEVICE, non_blocking=True), img_size=256)
        probs = None
        for turns in range(4):
            view = base if turns == 0 else torch.rot90(base, turns, dims=(-2, -1))
            for flipped in (view, view.flip(-1)):
                batch = flipped.contiguous(memory_format=torch.channels_last)
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(batch)
                piece = torch.softmax(logits.float(), dim=1)
                probs = piece if probs is None else probs + piece
        chunks.append((probs / 8).cpu())
    return torch.cat(chunks, dim=0)


def probs_of(state, images):
    model = make_model()
    model.load_state_dict(state)
    model.eval()
    out = tta_probs(model, images)
    del model
    torch.cuda.empty_cache()
    return out


def pair_acc(probs, labels):
    pred = probs.argmax(1).numpy()
    return float(accuracy_score(labels, pred)), pred


y_np = val_y.numpy()
p7_clean = probs_of(state7, val_u8)
p7_corrupt = probs_of(state7, corrupt_u8)
p42_clean = probs_of(state42, val_u8)
p42_corrupt = probs_of(state42, corrupt_u8)
acc7_c, _ = pair_acc(p7_clean, y_np)
acc7_k, _ = pair_acc(p7_corrupt, y_np)
acc42_c, _ = pair_acc(p42_clean, y_np)
acc42_k, _ = pair_acc(p42_corrupt, y_np)
ens_clean = (p7_clean + p42_clean) / 2
ens_corrupt = (p7_corrupt + p42_corrupt) / 2
acc_c, pred_c = pair_acc(ens_clean, y_np)
acc_k, pred_k = pair_acc(ens_corrupt, y_np)
print(f"F7 TTA8 {acc7_c:.4f}/{acc7_k:.4f}", flush=True)
print(f"F42 TTA8 {acc42_c:.4f}/{acc42_k:.4f}", flush=True)
print(f"ensemble TTA8 {acc_c:.4f}/{acc_k:.4f}", flush=True)

cm = confusion_matrix(y_np, pred_c, labels=list(range(20)))
fig, ax = plt.subplots(figsize=(10, 8))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(20), class_names, rotation=90, fontsize=7)
ax.set_yticks(range(20), class_names, fontsize=7)
ax.set_xlabel("predicted")
ax.set_ylabel("true")
ax.set_title("ensemble clean val")
fig.tight_layout()
plt.show()

pairs = []
for i in range(20):
    for j in range(20):
        if i != j and cm[i, j] > 0:
            pairs.append((int(cm[i, j]), i, j))
pairs.sort(reverse=True)
print("top-5 error pairs", flush=True)
for count, i, j in pairs[:5]:
    print(f"{class_names[i]} -> {class_names[j]} {count}", flush=True)

shown = []
for _count, i, j in pairs:
    hits = np.where((y_np == i) & (pred_c == j))[0]
    for idx in hits[:2]:
        shown.append(int(idx))
        if len(shown) == 6:
            break
    if len(shown) == 6:
        break
fig, axes = plt.subplots(2, 3, figsize=(9, 6))
for ax, idx in zip(axes.ravel(), shown):
    ax.imshow(val_u8[idx].permute(1, 2, 0).numpy())
    ax.set_title(f"{class_names[int(y_np[idx])]} -> {class_names[int(pred_c[idx])]}", fontsize=8)
    ax.axis("off")
for ax in axes.ravel()[len(shown):]:
    ax.axis("off")
fig.tight_layout()
plt.show()

## TTA и submission

Test загружается здесь и нужен только для предсказаний. TTA8 каждой модели, затем среднее softmax. Файл `submission.csv`: колонки id, label, 8299 строк, порядок как в test.csv. Рядом печатается доля совпадений с `outputs/submission_ens_F7F42_tta.csv`. Тот файл собран из чекпоинта F42 эпохи 11, а этот ноутбук берёт эпоху по среднему.

In [ ]:
test_table = pd.read_csv(tf.DATA_DIR / "test.csv", dtype={"id": str})
assert list(test_table.columns) == ["id"]
assert len(test_table) == N_TEST and test_table.id.is_unique
test_u8 = tf.load_uint8(test_table, "test")
assert tuple(test_u8.shape) == (N_TEST, 3, 256, 256)
test_probs = (probs_of(state7, test_u8) + probs_of(state42, test_u8)) / 2
labels = test_probs.argmax(1).numpy().astype(int)
submission = pd.DataFrame({"id": test_table.id.to_numpy(), "label": labels})
assert submission.id.tolist() == test_table.id.tolist()
assert len(submission) == N_TEST and submission.id.is_unique
assert submission.label.between(0, 19).all()
out_path = Path("submission.csv")
submission.to_csv(out_path, index=False, lineterminator="\n")
back = pd.read_csv(out_path, dtype={"id": str})
assert list(back.columns) == ["id", "label"] and len(back) == N_TEST
assert back.id.tolist() == test_table.id.tolist() and back.label.between(0, 19).all()
ref_path = Path("outputs/submission_ens_F7F42_tta.csv")
ref = pd.read_csv(ref_path, dtype={"id": str})
assert ref.id.tolist() == back.id.tolist()
same = int((ref.label.to_numpy() == back.label.to_numpy()).sum())
print(f"submission rows {len(back)} | agree with F7+F42 file {same}/{N_TEST} {same / N_TEST:.4f}", flush=True)

## Сравнение

MLP и CNN — базовые результаты задания. Остальные строки читаются из `outputs/experiments.csv`: одиночные F и E, ансамбли и отклонённые варианты.

In [ ]:
exp = pd.read_csv("outputs/experiments.csv", dtype=str).fillna("")

def lookup(pipeline, seed, variant, model="resnet50"):
    hit = exp[
        (exp.pipeline == pipeline)
        & (exp.model == model)
        & (exp.seed == seed)
        & (exp.variant == variant)
        & (exp.aug == "F")
    ]
    assert len(hit) == 1, (pipeline, seed, variant, model, len(hit))
    row = hit.iloc[0]
    clean = float(row.best_val_acc)
    corrupt = float(row.val_corrupt_acc)
    return clean, corrupt, (clean + corrupt) / 2

rows = [("MLP", 0.356, None, None), ("CNN", 0.549, None, None)]
spec = [
    ("F7 plain", "round3", "7", "plain", "resnet50"),
    ("F7 TTA8", "round2_tta", "7", "", "resnet50"),
    ("F42 plain", "round3", "42", "plain", "resnet50"),
    ("F42 TTA8", "ensemble_seed42_tta", "42", "", "resnet50"),
    ("E7 plain", "round4", "7", "E_plain", "resnet50"),
    ("E7 TTA8", "round4", "7", "E_plain_tta8", "resnet50"),
    ("E42 plain", "round4", "42", "E_plain", "resnet50"),
    ("E42 TTA8", "round4", "42", "E_plain_tta8", "resnet50"),
    ("ens F7+F42", "round4_ens", "7+42", "F7+F42", "resnet50"),
    ("ens F7+E7", "round4_ens", "7+7", "F7+E7", "resnet50"),
    ("ens F7+E42", "round4_ens", "7+42", "F7+E42", "resnet50"),
    ("ens E7+E42", "round4_ens", "7+42", "E7+E42", "resnet50"),
    ("ens 4 models", "round4_ens", "7+42+7+42", "F7+F42+E7+E42", "resnet50"),
    ("reject A 288", "round4", "7", "scale288_seed7", "resnet50"),
    ("reject A 320", "round4", "7", "scale320_seed7", "resnet50"),
    ("reject A multiscale", "round4", "7", "ms_tta_seed7", "resnet50"),
    ("reject B SWA", "round4", "7", "B_swa", "resnet50"),
    ("reject B EMA", "round4", "7", "B_ema", "resnet50"),
    ("reject C V2", "round4", "7", "C_plain", "resnet50"),
    ("reject D CutMix", "round4", "7", "D_plain", "resnet50"),
    ("reject F batch32", "round4", "7", "F_plain", "resnet50"),
    ("reject ResNet101", "round3", "7", "plain", "resnet101"),
    ("reject EMA 0.999", "round3", "7", "ema", "resnet50"),
]
print(f"{'run':22} {'clean':8} {'corrupt':8} {'mean':8}", flush=True)
for name, clean, corrupt, mean in rows:
    print(f"{name:22} {clean:8.4f} {'-':8} {'-':8}", flush=True)
for name, pipeline, seed, variant, model in spec:
    clean, corrupt, mean = lookup(pipeline, seed, variant, model)
    print(f"{name:22} {clean:8.4f} {corrupt:8.4f} {mean:8.4f}", flush=True)

## Время

Таймер стартовал в первой ячейке. Ожидание полного прогона около 34.5 минут: кэш, две серии по 15 эпох и TTA двух моделей на test. Если вышло больше 42 минут, ячейка печатает предупреждение.

In [ ]:
elapsed_min = (time.perf_counter() - PIPELINE_START) / 60.0
print(f"Elapsed time: {elapsed_min:.2f} minutes", flush=True)
if elapsed_min > 42:
    print("WARNING: elapsed time is above 42 minutes", flush=True)